# Image captioning benchmark: Gemini vs BLIP on COCO

Compares two captioning approaches on a fixed random sample of **COCO val2017** images:

| Model | Type |
|---|---|
| **Gemini Flash-Lite** (`gemini-3.1-flash-lite`) | General-purpose multimodal LLM, called through the API with a neutral prompt |
| **BLIP base** (`Salesforce/blip-image-captioning-base`) | Dedicated captioning model (ViT encoder + text decoder), run locally on the GPU |

**Metrics**

- **BLEU-4**: overlap of 1- to 4-word phrases with the 5 human reference captions, with a brevity penalty.
- **METEOR**: unigram matching that also accepts stems and synonyms, weighted towards recall.
- **CIDEr**: TF-IDF-weighted n-gram similarity to the references; rare, informative words count more. Designed for captioning.
- **CLIPScore**: reference-free; cosine similarity between the CLIP embeddings of the image and the caption (torchmetrics reports `100 × max(cos, 0)`).
- **Latency**: wall-clock time per caption (BLIP on the Colab GPU; Gemini over the network, so the two are not directly comparable).

The first three reward matching how COCO annotators phrase things; CLIPScore rewards describing the image correctly.

**How to run on Colab**

1. *Runtime → Change runtime type → T4 GPU*.
2. Add your Gemini key in the 🔑 *Secrets* panel as `GEMINI_API_KEY` and enable notebook access.
3. Run once with `SMOKE_TEST = True` (5 images) to check everything works, then set it to `False` and *Run all*.

Gemini captions are saved after every request. If the daily free quota runs out, re-run the notebook later
(the same or the next day) and it continues where it stopped. Set `USE_DRIVE = True` to keep that progress across Colab sessions; otherwise download `gemini_captions.json` before closing Colab.

Only measured numbers are reported; nothing is estimated.

In [ ]:
# ---- Configuration ----
SMOKE_TEST = True          # True: 5 images to test the pipeline. False: the full benchmark.
N_IMAGES = 150             # images in the full run (COCO val2017 has 5,000)
SEED = 42                  # fixes which images are sampled
USE_DRIVE = False          # True: keep data and cached Gemini captions on Google Drive between sessions

GEMINI_MODEL = "gemini-3.1-flash-lite"
GEMINI_RPM = 10            # requests per minute to send (stay under the free-tier limit)
# Same neutral, descriptive instruction for every image (no tone, matching COCO's plain style)
PROMPT = "Write a one-sentence caption that describes this image."

BLIP_MODEL = "Salesforce/blip-image-captioning-base"
BLIP_GENERATION = {"num_beams": 3, "max_length": 30, "min_length": 5}  # decoding settings from the BLIP paper

CLIP_MODEL = "openai/clip-vit-base-patch32"  # backbone used in the original CLIPScore paper

N_EXAMPLES = 3             # qualitative examples per direction of disagreement
BOOTSTRAP_SAMPLES = 2000   # resamples for the confidence intervals

In [ ]:
# ---- Setup: packages, Java (needed by the COCO tokenizer and METEOR), output folder ----
import importlib.util
import os
import shutil
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

REQUIRED = {"pycocoevalcap": "pycocoevalcap", "torchmetrics": "torchmetrics", "google.genai": "google-genai", "transformers": "transformers"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module.split(".")[0]) is None or importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

if shutil.which("java") is None:
    if IN_COLAB:
        print("Installing Java...")
        subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    else:
        raise RuntimeError("Java is required by pycocoevalcap (tokenizer and METEOR). Install a JRE and re-run.")

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/ai-captioner-benchmark"
else:
    BASE_DIR = os.path.abspath("benchmark_output")

RUN_DIR = os.path.join(BASE_DIR, "smoke" if SMOKE_TEST else "full")
DATA_DIR = os.path.join(BASE_DIR, "coco")
IMAGE_DIR = os.path.join(DATA_DIR, "val2017")
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
n_images = 5 if SMOKE_TEST else N_IMAGES
print(f"Run folder: {RUN_DIR}  |  images: {n_images}")

## 1. Sample COCO val2017 images and their reference captions

In [ ]:
import json
import random
import urllib.request
import zipfile

CAPTIONS_FILE = os.path.join(DATA_DIR, "captions_val2017.json")
ANNOTATIONS_ZIP_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

if not os.path.exists(CAPTIONS_FILE):
    print("Downloading COCO 2017 annotations (~240 MB, once)...")
    zip_path = os.path.join(DATA_DIR, "annotations.zip")
    urllib.request.urlretrieve(ANNOTATIONS_ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path) as archive, open(CAPTIONS_FILE, "wb") as out:
        out.write(archive.read("annotations/captions_val2017.json"))
    os.remove(zip_path)

with open(CAPTIONS_FILE) as f:
    coco = json.load(f)

file_names = {img["id"]: img["file_name"] for img in coco["images"]}
references = {}
for ann in coco["annotations"]:
    references.setdefault(ann["image_id"], []).append(ann["caption"].strip())

# Same seed -> same images on every run and machine
image_ids = random.Random(SEED).sample(sorted(file_names), n_images)

for image_id in image_ids:
    path = os.path.join(IMAGE_DIR, file_names[image_id])
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{file_names[image_id]}", path)

def image_path(image_id):
    return os.path.join(IMAGE_DIR, file_names[image_id])

print(f"{len(image_ids)} images, {sum(len(references[i]) for i in image_ids)} reference captions")
print("Example references:", references[image_ids[0]][:2])

## 2. BLIP captions (local GPU)

In [ ]:
import statistics
import time

import torch
from PIL import Image
from transformers import BlipForConditionalGeneration, BlipProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print("Device:", GPU_NAME)

blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL)
blip_model = BlipForConditionalGeneration.from_pretrained(BLIP_MODEL).to(device).eval()

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

@torch.no_grad()
def blip_caption(path):
    image = Image.open(path).convert("RGB")
    inputs = blip_processor(images=image, return_tensors="pt").to(device)
    output = blip_model.generate(**inputs, **BLIP_GENERATION)
    return blip_processor.decode(output[0], skip_special_tokens=True).strip()

blip_caption(image_path(image_ids[0]))  # warm-up, so model initialisation isn't counted as latency

blip_results = {}
for image_id in image_ids:
    sync()
    start = time.perf_counter()
    caption = blip_caption(image_path(image_id))
    sync()
    blip_results[image_id] = {"caption": caption, "latency_s": time.perf_counter() - start}

with open(os.path.join(RUN_DIR, "blip_captions.json"), "w") as f:
    json.dump({str(k): v for k, v in blip_results.items()}, f, indent=2)
print(f"BLIP: {len(blip_results)} captions, median {statistics.median(r['latency_s'] for r in blip_results.values()):.3f} s")
print("Example:", blip_results[image_ids[0]]["caption"])

## 3. Gemini captions (API, resumable)

Each caption is written to `gemini_captions.json` as soon as it arrives. Overloaded (5xx) and per-minute
rate-limit errors are retried with backoff; when the **daily** quota is exhausted the cell stops cleanly.
Re-run it later to continue.

In [ ]:
from google import genai
from google.genai import errors, types

def get_api_key():
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get("GEMINI_API_KEY")
        except Exception:
            pass
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("Set GEMINI_API_KEY (Colab: Secrets panel; elsewhere: environment variable).")
    return key

client = genai.Client(api_key=get_api_key())
GEMINI_CACHE = os.path.join(RUN_DIR, "gemini_captions.json")

gemini_results = {}
if os.path.exists(GEMINI_CACHE):
    with open(GEMINI_CACHE) as f:
        gemini_results = {int(k): v for k, v in json.load(f).items()}

def save_gemini_cache():
    tmp = GEMINI_CACHE + ".tmp"
    with open(tmp, "w") as f:
        json.dump({str(k): v for k, v in gemini_results.items()}, f, indent=2)
    os.replace(tmp, GEMINI_CACHE)

def clean_caption(text):
    return text.strip().strip("\"'“”").strip()

class DailyQuotaExhausted(Exception):
    pass

SERVER_BACKOFF_S = [5, 15, 30, 60]

def gemini_caption(path):
    """Returns (caption, latency of the successful request, attempts)."""
    with open(path, "rb") as f:
        image_bytes = f.read()
    contents = [types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"), PROMPT]
    config = types.GenerateContentConfig(temperature=0.0, max_output_tokens=100)
    attempts = 0
    while True:
        attempts += 1
        start = time.perf_counter()
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)
            latency = time.perf_counter() - start
            if not response.text:
                raise ValueError("empty response (possibly blocked)")
            return clean_caption(response.text), latency, attempts
        except errors.APIError as e:
            if e.code == 429:
                if "PerDay" in str(e):
                    raise DailyQuotaExhausted(str(e)[:300]) from e
                print("  per-minute limit hit, waiting 60 s")
                time.sleep(60)
            elif e.code >= 500 and attempts <= len(SERVER_BACKOFF_S):
                wait = SERVER_BACKOFF_S[attempts - 1]
                print(f"  {e.code} from Gemini, retrying in {wait} s")
                time.sleep(wait)
            else:
                raise

todo = [i for i in image_ids if i not in gemini_results]
print(f"Gemini: {len(gemini_results)} cached, {len(todo)} to go")
min_interval = 60 / GEMINI_RPM
try:
    for n, image_id in enumerate(todo, 1):
        started = time.perf_counter()
        try:
            caption, latency, attempts = gemini_caption(image_path(image_id))
            gemini_results[image_id] = {"caption": caption, "latency_s": latency, "attempts": attempts}
        except DailyQuotaExhausted:
            raise
        except Exception as e:  # e.g. a safety block on one image: record it and move on
            gemini_results[image_id] = {"caption": None, "error": str(e)[:300]}
            print(f"  image {image_id} failed: {str(e)[:120]}")
        save_gemini_cache()
        if n % 10 == 0 or n == len(todo):
            print(f"  {n}/{len(todo)} done")
        time.sleep(max(0.0, min_interval - (time.perf_counter() - started)))
except DailyQuotaExhausted as e:
    print("\nDaily Gemini quota exhausted. Progress is saved; re-run this notebook later to continue.")
    print(e)

done = [i for i in image_ids if gemini_results.get(i, {}).get("caption")]
print(f"Gemini captions available: {len(done)}/{len(image_ids)}")
if done:
    print("Example:", gemini_results[done[0]]["caption"])

## 4. Reference-based metrics: BLEU-4, METEOR, CIDEr (pycocoevalcap)

Evaluated on the images where **both** models produced a caption, so the comparison is paired.

In [ ]:
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

eval_ids = [i for i in image_ids if gemini_results.get(i, {}).get("caption") and blip_results[i]["caption"]]
if len(eval_ids) < len(image_ids):
    print(f"Note: evaluating {len(eval_ids)}/{len(image_ids)} images (the rest have no Gemini caption yet)")
assert eval_ids, "No images with captions from both models yet"

captions = {
    "BLIP": {i: blip_results[i]["caption"] for i in eval_ids},
    "Gemini": {i: gemini_results[i]["caption"] for i in eval_ids},
}

import traceback

import numpy as np

java_version = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0]
print("Java:", java_version)

tokenizer = PTBTokenizer()
gts = tokenizer.tokenize({i: [{"caption": c} for c in references[i]] for i in eval_ids})
tokenized = {m: tokenizer.tokenize({i: [{"caption": c[i]}] for i in eval_ids}) for m, c in captions.items()}
keys = list(gts.keys())  # pycocoevalcap returns per-image scores in this order

def meteor_coco(res):
    # Official COCO METEOR: a Java subprocess (meteor-1.5.jar) driven over stdin/stdout
    scorer = Meteor()
    try:
        score, scores = scorer.compute_score(gts, res)
    finally:
        scorer.meteor_p.kill()
    return float(score), dict(zip(keys, map(float, scores)))

def meteor_nltk(res):
    # Fallback: NLTK's pure-Python METEOR (WordNet synonyms). Corpus score = mean of per-image scores.
    import nltk
    from nltk.translate.meteor_score import meteor_score
    for resource in ["wordnet", "omw-1.4"]:
        nltk.download(resource, quiet=True)
    scores = {i: float(meteor_score([r.split() for r in gts[i]], res[i][0].split())) for i in keys}
    return float(np.mean(list(scores.values()))), scores

METEOR_IMPL = None
metrics, per_image = {}, {}
for model_name, res in tokenized.items():
    bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    cider, cider_scores = Cider().compute_score(gts, res)
    metrics[model_name] = {"BLEU-4": float(bleu[3]), "CIDEr": float(cider)}
    per_image[model_name] = {"CIDEr": dict(zip(keys, map(float, cider_scores)))}

    # Use the same METEOR implementation for both models, so the comparison stays fair
    for impl_name, impl in [("pycocoevalcap (meteor-1.5.jar)", meteor_coco), ("nltk", meteor_nltk)]:
        if METEOR_IMPL not in (None, impl_name):
            continue
        try:
            metrics[model_name]["METEOR"], per_image[model_name]["METEOR"] = impl(res)
            METEOR_IMPL = impl_name
            break
        except Exception:
            print(f"METEOR via {impl_name} failed:")
            traceback.print_exc()
    print(model_name, {k: round(v, 4) for k, v in metrics[model_name].items()})

if all("METEOR" in metrics[m] for m in metrics):
    print("METEOR implementation:", METEOR_IMPL)
else:
    METEOR_IMPL = None
    for m in metrics:
        metrics[m].pop("METEOR", None)
        per_image[m].pop("METEOR", None)
    print("METEOR unavailable; continuing with BLEU-4, CIDEr and CLIPScore")

## 5. Reference-free metric: CLIPScore (torchmetrics)

In [ ]:
from torchmetrics.multimodal.clip_score import CLIPScore

clip_metric = CLIPScore(model_name_or_path=CLIP_MODEL).to(device)

def clip_score(path, caption):
    image = torch.from_numpy(np.array(Image.open(path).convert("RGB"))).permute(2, 0, 1)
    clip_metric.reset()
    clip_metric.update(image.unsqueeze(0).to(device), [caption])
    return float(clip_metric.compute())

for model_name, model_captions in captions.items():
    scores = {i: clip_score(image_path(i), model_captions[i]) for i in eval_ids}
    per_image[model_name]["CLIPScore"] = scores
    metrics[model_name]["CLIPScore"] = float(np.mean(list(scores.values())))
    print(model_name, "CLIPScore:", round(metrics[model_name]["CLIPScore"], 2))

ref_lengths = [len(c.split()) for i in eval_ids for c in references[i]]
for model_name, model_captions in captions.items():
    lengths = [len(model_captions[i].split()) for i in eval_ids]
    metrics[model_name]["avg_words"] = float(np.mean(lengths))
print("Reference captions average", round(float(np.mean(ref_lengths)), 1), "words")

## 6. Latency, paired bootstrap confidence intervals, and win rate

The bootstrap resamples images with replacement and recomputes the difference in mean per-image score
(Gemini − BLIP). If the 95% interval excludes 0, the difference is unlikely to be chance on this sample.
(CIDEr's IDF weights are held fixed from the full sample: a standard approximation.)

In [ ]:
def latency_stats(results):
    values = sorted(results[i]["latency_s"] for i in eval_ids)
    return {
        "mean_s": float(np.mean(values)),
        "median_s": float(np.median(values)),
        "p95_s": float(np.percentile(values, 95)),
    }

metrics["BLIP"]["latency"] = latency_stats(blip_results)
metrics["Gemini"]["latency"] = latency_stats(gemini_results)
gemini_retries = sum(gemini_results[i].get("attempts", 1) - 1 for i in eval_ids)
gemini_failed = sum(1 for i in image_ids if i in gemini_results and not gemini_results[i].get("caption"))

rng = np.random.default_rng(SEED)
comparisons = {}
SCORED_METRICS = [m for m in ["CIDEr", "METEOR", "CLIPScore"] if m in per_image["Gemini"]]
for metric_name in SCORED_METRICS:
    g = np.array([per_image["Gemini"][metric_name][i] for i in eval_ids])
    b = np.array([per_image["BLIP"][metric_name][i] for i in eval_ids])
    diffs = g - b
    boot = [diffs[rng.integers(0, len(diffs), len(diffs))].mean() for _ in range(BOOTSTRAP_SAMPLES)]
    low, high = np.percentile(boot, [2.5, 97.5])
    comparisons[metric_name] = {
        "mean_diff_gemini_minus_blip": float(diffs.mean()),
        "ci95": [float(low), float(high)],
        "gemini_win_rate": float((diffs > 0).mean()),
    }
    print(f"{metric_name:9s} Δ(Gemini−BLIP) = {diffs.mean():+.4f}  95% CI [{low:+.4f}, {high:+.4f}]  Gemini better on {(diffs > 0).mean():.0%} of images")

## 7. Save results: `results.json`, `results.md`, `examples.md`, `captions.json`

In [ ]:
import datetime
import platform

import transformers

results = {
    "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "smoke_test": SMOKE_TEST,
    "dataset": "COCO val2017",
    "seed": SEED,
    "n_sampled": len(image_ids),
    "n_evaluated": len(eval_ids),
    "models": {
        "Gemini": {"id": GEMINI_MODEL, "prompt": PROMPT, "temperature": 0.0},
        "BLIP": {"id": BLIP_MODEL, "generation": BLIP_GENERATION},
    },
    "clipscore_model": CLIP_MODEL,
    "meteor_implementation": METEOR_IMPL,
    "metrics": metrics,
    "paired_comparison": comparisons,
    "gemini": {"retries": gemini_retries, "failed_images": gemini_failed, "requests_per_minute_cap": GEMINI_RPM},
    "reference_avg_words": float(np.mean(ref_lengths)),
    "environment": {
        "device": GPU_NAME,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    },
}
with open(os.path.join(RUN_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

def fmt_ci(c):
    return f"{c['mean_diff_gemini_minus_blip']:+.3f} [{c['ci95'][0]:+.3f}, {c['ci95'][1]:+.3f}]"

lines = [
    f"# Captioning benchmark results ({'smoke test' if SMOKE_TEST else 'full run'})",
    "",
    f"COCO val2017, {len(eval_ids)} images sampled with seed {SEED}, 5 human reference captions each. "
    f"Generated {results['generated_at']} on {GPU_NAME}.",
    "",
    "| Model | BLEU-4 | METEOR | CIDEr | CLIPScore | Avg. words | Median latency | p95 latency |",
    "|---|---|---|---|---|---|---|---|",
]
for model_name in ["Gemini", "BLIP"]:
    m = metrics[model_name]
    model_id = results["models"][model_name]["id"]
    meteor_cell = f"{m['METEOR']:.3f}" if "METEOR" in m else "n/a"
    lines.append(
        f"| {model_name} (`{model_id}`) | {m['BLEU-4']:.3f} | {meteor_cell} | {m['CIDEr']:.3f} | {m['CLIPScore']:.2f} "
        f"| {m['avg_words']:.1f} | {m['latency']['median_s']:.2f} s | {m['latency']['p95_s']:.2f} s |"
    )
lines += [
    "",
    f"Human reference captions average {results['reference_avg_words']:.1f} words. METEOR implementation: {METEOR_IMPL or 'unavailable'}.",
    "",
    "**Paired difference (Gemini − BLIP), mean per image with 95% bootstrap CI**",
    "",
    "| Metric | Δ [95% CI] | Gemini better on |",
    "|---|---|---|",
]
for metric_name, c in comparisons.items():
    lines.append(f"| {metric_name} | {fmt_ci(c)} | {c['gemini_win_rate']:.0%} of images |")
lines += [
    "",
    f"BLIP latency is local GPU inference ({GPU_NAME}); Gemini latency includes the network round trip "
    f"(free tier, {gemini_retries} retried requests, {gemini_failed} failed images).",
]
with open(os.path.join(RUN_DIR, "results.md"), "w") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))

# Qualitative examples: images where the models disagree most on CIDEr
cider_diff = {i: per_image["Gemini"]["CIDEr"][i] - per_image["BLIP"]["CIDEr"][i] for i in eval_ids}
ranked = sorted(eval_ids, key=cider_diff.get)
sections = [("BLIP scores much higher", ranked[:N_EXAMPLES]), ("Gemini scores much higher", ranked[::-1][:N_EXAMPLES])]
ex = ["# Where the models disagree most (per-image CIDEr)", ""]
for title, ids in sections:
    ex += [f"## {title}", ""]
    for i in ids:
        ex += [
            f"### COCO image {i}",
            "",
            f"![COCO {i}](http://images.cocodataset.org/val2017/{file_names[i]})",
            "",
            f"- **Gemini:** {captions['Gemini'][i]}  (CIDEr {per_image['Gemini']['CIDEr'][i]:.2f}, CLIPScore {per_image['Gemini']['CLIPScore'][i]:.1f})",
            f"- **BLIP:** {captions['BLIP'][i]}  (CIDEr {per_image['BLIP']['CIDEr'][i]:.2f}, CLIPScore {per_image['BLIP']['CLIPScore'][i]:.1f})",
            "- **References:**",
            *[f"  - {r}" for r in references[i]],
            "",
        ]
with open(os.path.join(RUN_DIR, "examples.md"), "w") as f:
    f.write("\n".join(ex))

with open(os.path.join(RUN_DIR, "captions.json"), "w") as f:
    json.dump(
        [
            {
                "image_id": i,
                "file_name": file_names[i],
                "references": references[i],
                **{
                    model_name.lower(): {
                        "caption": captions[model_name][i],
                        **{k: per_image[model_name][k][i] for k in SCORED_METRICS},
                    }
                    for model_name in captions
                },
            }
            for i in eval_ids
        ],
        f,
        indent=2,
    )
print("\nSaved to", RUN_DIR, ":", sorted(os.listdir(RUN_DIR)))

## 8. Bring the results back to the repo

Download `results.json`, `results.md`, `examples.md` and `captions.json` from the run folder
(on Drive: `MyDrive/ai-captioner-benchmark/full/`) into the repo's `benchmark/results/` folder and commit them.